# parte 4 - aprendizagem nao supervisionada
## 4.c analise de associacao (apriori) e 4.d deteccao de outliers (lof)

neste notebook vamos usar o apriori para encontrar regras de associacao
entre as caracteristicas das casas e o lof para identificar outliers.

In [ ]:
# importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_rows', 200)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# carregando os dados
df = pd.read_csv('train.csv')

## repetindo as transformacoes que o felipe fez

In [ ]:
# target encoding no bairro
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['Neighborhood'])
df['Neighborhood'] = encoder.fit_transform(df['Neighborhood'], df['SalePrice'])

In [ ]:
# transformando variaveis categoricas em numeros
colunas_ordinais = ['ExterQual', 'KitchenQual', 'BsmtQual']
mapeamento = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1}

df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento)

df['BsmtQual'] = df['BsmtQual'].fillna('Po')
mapeamento_completo = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento_completo)

In [ ]:
# criando as features novas
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
df['TotalArea'] = df['GrLivArea'] + df['TotalBsmtSF']
df['TotalBath'] = (df['FullBath'] + 0.5*df['HalfBath']
                   + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

In [ ]:
# selecionando as mesmas features
features = [
    'OverallQual', 'TotalArea', 'Neighborhood', 'ExterQual',
    'KitchenQual', 'GarageCars', 'TotalBath', 'BsmtQual',
    'HouseAge', 'YearsSinceRemodel'
]

X = df[features].fillna(0)

# padronizando
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4.c - analise de associacao com apriori

vamos transformar as features numericas em categorias
para encontrar regras do tipo "se x entao y".

In [ ]:
# transformando as features numericas em categorias
df_apriori = df.copy()

# criando categorias para cada feature
df_apriori['OverallQual_cat'] = pd.cut(df['OverallQual'], bins=3,
    labels=['qualidade_baixa', 'qualidade_media', 'qualidade_alta'])

df_apriori['TotalArea_cat'] = pd.qcut(df['TotalArea'], q=3,
    labels=['area_pequena', 'area_media', 'area_grande'])

df_apriori['GarageCars_cat'] = df['GarageCars'].astype(str).replace(
    {'0': 'sem_garagem', '1': '1_vaga', '2': '2_vagas', '3': '3_vagas', '4': '4+_vagas'})

df_apriori['HouseAge_cat'] = pd.cut(df['HouseAge'], bins=3,
    labels=['casa_nova', 'casa_media', 'casa_antiga'])

df_apriori['SalePrice_cat'] = pd.qcut(df['SalePrice'], q=2,
    labels=['preco_baixo', 'preco_alto'])

df_apriori['TotalBath_cat'] = pd.cut(df['TotalBath'], bins=3,
    labels=['poucos_banheiros', 'banheiros_medio', 'muitos_banheiros'])

df_apriori['ExterQual_cat'] = df['ExterQual'].map(
    {0: 'exterior_pobre', 1: 'exterior_fraco', 2: 'exterior_medio',
     3: 'exterior_bom', 4: 'exterior_excelente'})

# mostrando como ficou
print(df_apriori[['OverallQual_cat', 'TotalArea_cat', 'GarageCars_cat',
                  'HouseAge_cat', 'SalePrice_cat']].head())

In [ ]:
# criando a tabela de transacoes (formato one-hot)
colunas_categoricas = [
    'OverallQual_cat', 'TotalArea_cat', 'GarageCars_cat',
    'HouseAge_cat', 'SalePrice_cat', 'TotalBath_cat', 'ExterQual_cat'
]

transacoes = pd.get_dummies(df_apriori[colunas_categoricas])

print('formato da tabela de transacoes:', transacoes.shape)
print()
print(transacoes.head())

In [ ]:
# rodando o apriori para encontrar itens frequentes
itens_frequentes = apriori(transacoes, min_support=0.1, use_colnames=True)

print('itens frequentes encontrados:')
print(itens_frequentes.sort_values('support', ascending=False).head(15))

In [ ]:
# gerando as regras de associacao
regras = association_rules(itens_frequentes, metric='confidence', min_threshold=0.5)

# ordenando pelas melhores metricas
regras = regras.sort_values('lift', ascending=False)

print(f'total de regras encontradas: {len(regras)}')
print()
print('top 10 regras (ordenadas por lift):')
print(regras[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

In [ ]:
# filtrando regras que tem a ver com preco alto ou baixo
regras_preco = regras[regras['consequents'].apply(
    lambda x: 'preco_alto' in str(x) or 'preco_baixo' in str(x))]

print('regras relacionadas ao preco:')
print(regras_preco[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string())

### interpretacao do apriori

as regras mostram combinacoes de caracteristicas que aparecem juntas.
por exemplo, se uma casa tem qualidade alta e area grande,
e muito provavel que o preco tambem seja alto.

o **lift** mostra o quanto a regra e mais forte do que o acaso.
quanto maior o lift, mais forte e a associacao.

## 4.d - deteccao de outliers com local outlier factor (lof)

vamos usar o lof para encontrar casas que sao diferentes da maioria.

In [ ]:
# aplicando o lof para detectar outliers
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
outliers = lof.fit_predict(X_scaled)

# -1 = outlier, 1 = normal
df['outlier'] = outliers

qtde_outliers = (outliers == -1).sum()
print(f'total de outliers encontrados: {qtde_outliers} ({qtde_outliers/len(df):.1%} dos dados)')

In [ ]:
# comparando estatisticas entre normais e outliers
colunas_comparar = features + ['SalePrice']
comparacao = df.groupby('outlier')[colunas_comparar].mean().round(2)

print('comparacao entre casas normais e outliers:')
print(comparacao)
print()
print('quantidade:')
print(df['outlier'].value_counts())

In [ ]:
# visualizando os outliers com pca
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))

# normais em azul
normais = df['outlier'] == 1
plt.scatter(X_pca[normais, 0], X_pca[normais, 1],
            c='blue', label='normal', alpha=0.5, s=50)

# outliers em vermelho
outliers_mask = df['outlier'] == -1
plt.scatter(X_pca[outliers_mask, 0], X_pca[outliers_mask, 1],
            c='red', label='outlier', alpha=0.8, s=80, edgecolors='k')

plt.xlabel('componente principal 1')
plt.ylabel('componente principal 2')
plt.title('outliers detectados pelo lof (visualizacao com pca)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# mostrando alguns exemplos de outliers
outliers_df = df[df['outlier'] == -1].copy()

print('exemplos de casas consideradas outliers:')
print(outliers_df[features + ['SalePrice']].head(10))

In [ ]:
# distribuicao dos scores de outlier
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(lof.negative_outlier_factor_, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(x=-1, color='r', linestyle='--', label='limite (score = -1)')
plt.xlabel('score de outlier')
plt.ylabel('frequencia')
plt.title('distribuicao dos scores lof')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([lof.negative_outlier_factor_[normais],
             lof.negative_outlier_factor_[outliers_mask]],
            labels=['normal', 'outlier'])
plt.ylabel('score de outlier')
plt.title('boxplot dos scores por grupo')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# distribuicao dos scores de outlier
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(lof.negative_outlier_factor_, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(x=-1, color='r', linestyle='--', label='limite (score = -1)')
plt.xlabel('score de outlier')
plt.ylabel('frequencia')
plt.title('distribuicao dos scores lof')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([lof.negative_outlier_factor_[normais],
             lof.negative_outlier_factor_[outliers_mask]],
            labels=['normal', 'outlier'])
plt.ylabel('score de outlier')
plt.title('boxplot dos scores por grupo')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### conclusao

com o apriori encontramos regras que mostram quais combinacoes
de caracteristicas levam a precos altos ou baixos.

com o lof identificamos as casas mais atipicas do dataset,
que podem ser casos interessantes para analisar mais a fundo.